# Домашнее задание 3. Экстрактор структурных блоков

Здравствуйте! Проект продолжается.

> **Пятница, 11:20. Третье письмо заказчика.**
> «Классификатор работает, спасибо. Но нам мало знать, что это счет — нужно достать из него таблицу с позициями. Сейчас бухгалтер открывает документ и перепечатывает цифры руками. Можно ли научить систему находить таблицу на странице и вырезать ее отдельной картинкой? Мы заказали у подрядчика разметку — шестьсот страниц с обведенными блоками, прикладываю.»

Задача этого задания: научиться отвечать не на вопрос: «Что это за документ?», а на вопрос: **«Где на нем что находится?»** — и вырезать найденное.

### Что нужно положить в `/content` перед началом
1. **`dz3_pages.zip`** — размеченные страницы от подрядчика (скачайте с LMS, около 180 МБ).
2. **`dz2_artifacts.zip`** — **ваш** архив из ДЗ 2: понадобится обученный классификатор.
3. **`dataset_dz1_raw.zip`** — архив сканов заказчика (с LMS): на нем будем проверять, переносится ли модель.

**Включите видеокарту:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**. Сегментация без нее считается неприлично долго.

### Как устроен ноутбук
- **Инфраструктурные ячейки** работают сами — просто запускайте по порядку.
- **Ячейки с пометкой `TODO`** — ваша работа: каркас дан, ключевые строки заменены на `...` с наводящим вопросом.
- **Ячейки-самопроверки ✅** после каждого шага говорят, что шаг пройден.
- **Отчетные ячейки ✍️** — три строки своими словами.
- **Ячейки «попробуйте сами» 🧪** — необязательные эксперименты.

### Связь с дисциплиной
Шаг 1 — пары «изображение-маска» и синхронные аугментации (3.1), Шаг 2 — архитектура U-Net и первый запуск (3.1), Шаг 3 — метрики IoU и Dice (3.2), Шаг 4 — Dice Loss против дисбаланса (3.3), Шаг 5 — Segment Anything без дообучения (3.4), Шаг 6 — вырезание области интереса (3.5), Шаг 7 — перенос на данные заказчика.

**Ориентир трудоемкости: 5-6 часов**, из них обучение — около двадцати минут машинного времени.

### Что будет на выходе
Архив **`dz3_artifacts.zip`**: веса U-Net, явный `model_config.json`, самодостаточный `table_extractor.pt`, метрики, вырезанные таблицы, сравнение с SAM и журнал экспериментов. Сохраните его — с него начнется ДЗ 4.

## Данные и окружение

In [ ]:
# --- Распаковка трех архивов (инфраструктура) ---
import os, zipfile, shutil
from pathlib import Path

CONTENT = Path("/content")
REQUIRED = {"dz3_pages.zip": CONTENT / "pages",
            "dz2_artifacts.zip": CONTENT / "dz2",
            "dataset_dz1_raw.zip": CONTENT / "archive"}

missing = [n for n in REQUIRED if not (CONTENT / n).exists()]
if missing:
    raise FileNotFoundError(
        "В /content не найдены файлы: " + ", ".join(missing) + ". "
        "Скачайте их с LMS (архив артефактов — ваш, из ДЗ 2) и перетащите в панель «Файлы» слева."
    )

for name, target in REQUIRED.items():
    zip_path = CONTENT / name
    if not zipfile.is_zipfile(zip_path):
        raise RuntimeError(f"{name} не читается как zip — вероятно, загрузка не завершилась.")
    if target.exists() and zip_path.stat().st_mtime <= target.stat().st_mtime:
        print(f"✅ {name}: уже распакован")
    else:
        shutil.rmtree(target, ignore_errors=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(target)
        print(f"✅ {name} ({zip_path.stat().st_size / 1e6:.1f} МБ) распакован")

PAGES = next(p.parent for p in (CONTENT / "pages").rglob("images") if p.is_dir())
DZ2 = next(p.parent for p in (CONTENT / "dz2").rglob("metrics.json"))
ARCHIVE = CONTENT / "archive"

n_pages = len(os.listdir(PAGES / "images"))
n_masks = len(os.listdir(PAGES / "masks"))
print()
print(f"размеченных страниц: {n_pages}, масок: {n_masks}")
print("артефакты ДЗ 2:", sorted(p.name for p in DZ2.iterdir()))


In [ ]:
# --- Библиотеки, случайность, устройство (инфраструктура) ---
import sys, subprocess
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'albumentations==1.4.*', 'timm==1.0.*'
])

import os, logging, warnings
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
warnings.filterwarnings('ignore', message='.*HF_TOKEN.*')
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)

import random, json, time, copy
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, cv2
import matplotlib.pyplot as plt
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from torchvision import models as tv_models
import timm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = 384
EPOCHS = 25 if device == 'cuda' else 4
PATIENCE = 6

print('устройство:', device, '| размер входа:', IMG_SIZE, '| максимум эпох:', EPOCHS)
if device == 'cpu':
    print('⚠️ Видеокарта не подключена: «Среда выполнения» → «Сменить среду выполнения» → T4 GPU.')
else:
    print('видеокарта:', torch.cuda.get_device_name(0))


## ✍️ Рефлексия ДО старта

**1. Чем задача «найти таблицу на странице» принципиально отличается от задачи «определить тип документа»?** Что теперь является ответом модели?

✍️ Ваш ответ: ...

**2. Заказчик просит вырезать таблицы. Какой результат вы считаете хорошим?** Попробуйте сформулировать измеримо.

✍️ Ваш ответ: ...

**3. Таблицы есть далеко не на каждой странице.** Как, по-вашему, это повлияет на обучение?

✍️ Ваш ответ: ...

## Шаг 1. Пары «страница-маска» и синхронные аугментации

Единица данных теперь двойная: страница и ее маска. Маска — это изображение того же размера, где вместо яркости записан номер области: 0 — фон, 1 — заголовок, 2 — текст, 3 — таблица, 4 — картинка.

Мы решаем **бинарную** задачу: таблица или не таблица. Из многоклассовой маски берем только значение 3 и превращаем ее в маску из нулей и единиц.

Два правила работы с масками, которые легко нарушить:

- **Ресайз только ближайшим соседом.** Обычная интерполяция усредняет соседние пиксели, и на границе появится значение 0.37 — маска перестанет быть маской.
- **Аугментации только синхронные.** Повернули страницу — обязаны повернуть и маску тем же поворотом, иначе разметка «уедет» и модель будет учиться на лжи.

In [ ]:
# --- TODO 1.1: датасет пар и синхронные аугментации ---
TABLE_VALUE = 3          # какое значение в многоклассовой маске означает таблицу

all_pages = sorted(p.stem for p in (PAGES / "images").glob("*.jpg"))
rng = random.Random(SEED)
shuffled = all_pages.copy(); rng.shuffle(shuffled)
cut = int(0.8 * len(shuffled))
train_pages, val_pages = sorted(shuffled[:cut]), sorted(shuffled[cut:])
print(f"страниц: обучение {len(train_pages)}, валидация {len(val_pages)}")

# TODO 1.1 - соберите два набора преобразований.
#   Albumentations умеет применять одно и то же преобразование к паре: transform(image=..., mask=...).
#   Для обучения: приведение к IMG_SIZE, небольшой поворот, изменение яркости и контраста.
#   Для валидации: только приведение к размеру, ничего случайного.
#   ВНИМАНИЕ на два момента:
#     1) у A.Resize есть параметр mask_interpolation — каким методом ресайзить маску?
#        Подумайте, что произойдет с маской при обычной билинейной интерполяции;
#     2) поворот тоже интерполирует маску: у A.Rotate есть отдельный параметр для маски.
#   Подсказка по именам: mask_interpolation=cv2.INTER_NEAREST.

def build_train_aug():
    return ...

def build_val_aug():
    return ...

train_tf, val_tf = build_train_aug(), build_val_aug()
if train_tf is Ellipsis or val_tf is Ellipsis:
    raise NotImplementedError("Заполните TODO 1.1: два пайплайна преобразований.")


class PageMaskDataset(Dataset):
    """Пара «страница — бинарная маска таблиц»."""

    def __init__(self, page_names, transform):
        self.names = page_names
        self.transform = transform

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        image = cv2.imread(str(PAGES / "images" / f"{name}.jpg"), cv2.IMREAD_GRAYSCALE)
        full_mask = cv2.imread(str(PAGES / "masks" / f"{name}.png"), cv2.IMREAD_GRAYSCALE)
        mask = (full_mask == TABLE_VALUE).astype(np.uint8)      # многоклассовая -> бинарная
        out = self.transform(image=image, mask=mask)
        img_t = torch.from_numpy(out["image"]).float().unsqueeze(0) / 255.0
        mask_t = torch.from_numpy(out["mask"]).float().unsqueeze(0)
        return img_t, mask_t


train_ds = PageMaskDataset(train_pages, train_tf)
val_ds = PageMaskDataset(val_pages, val_tf)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

xb, yb = next(iter(train_loader))
print(f"батч: страницы {tuple(xb.shape)}, маски {tuple(yb.shape)}")
print(f"значения в маске: {sorted(torch.unique(yb).tolist())}")


In [ ]:
# --- Смотрим глазами: страница, маска и наложение (инфраструктура) ---
with_table = [n for n in train_pages
              if (cv2.imread(str(PAGES / "masks" / f"{n}.png"), cv2.IMREAD_GRAYSCALE) == TABLE_VALUE).any()][:3]
fig, axes = plt.subplots(3, len(with_table), figsize=(4 * len(with_table), 11))
for j, name in enumerate(with_table):
    image = cv2.imread(str(PAGES / "images" / f"{name}.jpg"), cv2.IMREAD_GRAYSCALE)
    mask = (cv2.imread(str(PAGES / "masks" / f"{name}.png"), cv2.IMREAD_GRAYSCALE) == TABLE_VALUE)
    axes[0, j].imshow(image, cmap="gray"); axes[0, j].axis("off"); axes[0, j].set_title(name, fontsize=9)
    axes[1, j].imshow(mask, cmap="gray"); axes[1, j].axis("off"); axes[1, j].set_title("маска таблиц", fontsize=9)
    overlay = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours, -1, (220, 40, 40), 5)      # контур, а не заливка: видно текст под ним
    axes[2, j].imshow(overlay); axes[2, j].axis("off")
    axes[2, j].set_title("контур маски поверх страницы", fontsize=9)
plt.tight_layout(); plt.show()

# доля площади таблиц и доля пустых масок
shares, empties = [], 0
for name in train_pages:
    m = (cv2.imread(str(PAGES / "masks" / f"{name}.png"), cv2.IMREAD_GRAYSCALE) == TABLE_VALUE)
    shares.append(float(m.mean()))
    empties += int(not m.any())
shares = np.array(shares)
print(f"таблицы занимают в среднем {shares.mean():.1%} площади страницы")
print(f"страниц без таблиц вовсе: {empties} из {len(train_pages)} ({empties / len(train_pages):.0%})")
print()
print("Запомните эти два числа: первое — про дисбаланс, второе — про деление на ноль в метриках.")


In [ ]:
# --- ✅ Самопроверка Шага 1 ---
assert not (set(train_pages) & set(val_pages)), "Выборки пересекаются."
assert xb.shape[-2:] == (IMG_SIZE, IMG_SIZE), f"Страница должна быть {IMG_SIZE}x{IMG_SIZE}"
assert yb.shape == xb.shape, "Маска должна быть того же размера, что и страница."
unique_vals = set(torch.unique(yb).tolist())
assert unique_vals <= {0.0, 1.0}, \
    (f"В маске встретились значения {sorted(unique_vals)}. Маска обязана состоять только из 0 и 1 - "
     "проверьте, каким методом интерполируется маска при ресайзе и повороте.")

# проверка синхронности: сдвигаем пару и убеждаемся, что маска поехала вместе с картинкой
probe_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE, mask_interpolation=cv2.INTER_NEAREST),
                      A.Rotate(limit=(20, 20), interpolation=cv2.INTER_LINEAR,
                               mask_interpolation=cv2.INTER_NEAREST,
                               border_mode=cv2.BORDER_CONSTANT, value=255, mask_value=0, p=1.0)])
sample_name = with_table[0]
img0 = cv2.imread(str(PAGES / "images" / f"{sample_name}.jpg"), cv2.IMREAD_GRAYSCALE)
msk0 = (cv2.imread(str(PAGES / "masks" / f"{sample_name}.png"), cv2.IMREAD_GRAYSCALE) == TABLE_VALUE).astype(np.uint8)
rotated = probe_tf(image=img0, mask=msk0)
dark_in_mask_before = float(img0[msk0 > 0].mean())
dark_in_mask_after = float(rotated["image"][rotated["mask"] > 0].mean())
assert abs(dark_in_mask_before - dark_in_mask_after) < 25, \
    ("После поворота маска перестала совпадать с содержимым страницы. "
     "Похоже, преобразование применяется к изображению и маске порознь.")

print(f"✅ Шаг 1 пройден: маска бинарная, преобразования синхронные.")
print(f"Таблицы занимают {shares.mean():.1%} площади, пустых масок {empties / len(train_pages):.0%}.")


### ✍️ Отчетная ячейка Шага 1
1) **Каким методом я делаю ресайз маски и почему** (что было бы при обычной интерполяции): ...
2) **Как я убедился, что аугментации синхронные**: ...
3) **Два числа про данные** — доля площади таблиц и доля пустых масок. Что они предвещают для обучения: ...

## Шаг 2. U-Net и первый запуск
Классификатор из ДЗ 2 сжимал изображение до одного ответа. Здесь ответ должен быть размером со страницу — значит, после сжатия нужно вернуть разрешение обратно.

U-Net так и устроена: энкодер сжимает (понимает «что»), декодер восстанавливает (возвращает «где»), а **пропускные связи** передают детали с ранних слоев напрямую в декодер. Без них границы получаются мыльными: информация о точном положении теряется при сжатии и восстановить ее из одного «дна» невозможно.

Ниже — каркас сети. Ваша задача: соединить половины.

In [ ]:
# --- TODO 2.1: соберите U-Net ---
class DoubleConv(nn.Module):
    """Два сверточных слоя подряд — базовый кирпич U-Net."""
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.enc1 = DoubleConv(1, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.bottleneck = DoubleConv(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)          # вход вдвое шире: см. TODO
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, 1, kernel_size=1)       # один канал: таблица или нет

    def forward(self, x):
        skip1 = self.enc1(x)                                 # сохраняем для пропускной связи
        skip2 = self.enc2(self.pool(skip1))
        skip3 = self.enc3(self.pool(skip2))
        bottom = self.bottleneck(self.pool(skip3))

        # TODO 2.1 — три строки, по одной на каждый уровень декодера.
        #   Схема одна и та же: увеличить карту (self.upN), СОЕДИНИТЬ ее с сохраненным skipN
        #   и пропустить через self.decN.
        #   Вопрос, на который надо ответить: как именно соединить две карты признаков?
        #   Обратите внимание на размерности: dec3 ждет на входе base*8 каналов,
        #   а up3 выдает base*4 и skip3 содержит base*4. Подсказка: torch.cat и ось каналов.
        x = ...
        x = ...
        x = ...
        if x is Ellipsis:
            raise NotImplementedError(
                "Заполните TODO 2.1: соедините увеличенную карту с сохраненным skip и пропустите через dec.")
        return self.head(x)


model = UNet().to(device)
probe_out = model(xb[:2].to(device))
if probe_out is Ellipsis or probe_out.shape[-2:] != (IMG_SIZE, IMG_SIZE):
    raise NotImplementedError("Заполните TODO 2.1: соедините энкодер и декодер пропускными связями.")

print(f"параметров в сети: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} млн")
print(f"вход {tuple(xb[:2].shape)} -> выход {tuple(probe_out.shape)}")
print("выход того же размера, что и вход: сеть предсказывает ответ в каждом пикселе")


### От логитов к маске

Сеть выдает не маску, а **логиты**: по одному сырому числу на пиксель. Чем больше число, тем сильнее сеть верит, что здесь таблица. Чтобы получить маску, нужны два шага: сигмоида переводит логит в вероятность от 0 до 1, порог превращает вероятность в решение.

Первую функцию потерь берем привычную — бинарную кросс-энтропию; ту же, что в классификации, только примененную к каждому пикселю. В PyTorch для этого есть `BCEWithLogitsLoss`: сигмоида уже внутри, подавать нужно логиты.

In [ ]:
# --- Цикл обучения с ранней остановкой (инфраструктура) ---
def pixel_accuracy(logits, target, threshold=0.5):
    pred = (torch.sigmoid(logits) > threshold).float()
    return float((pred == target).float().mean())


def fit(model, name, loss_fn, epochs=None, lr=1e-3, patience=None):
    epochs = epochs or EPOCHS
    patience = patience or PATIENCE
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_dice": []}
    best_score, best_state, best_epoch, waited = -1.0, None, 0, 0
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        running, seen = 0.0, 0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = loss_fn(logits, masks)
            loss.backward(); optimizer.step()
            running += loss.item() * images.size(0); seen += images.size(0)
        train_loss = running / seen

        model.eval()
        v_loss, v_seen, accs, dices = 0.0, 0, [], []
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                logits = model(images)
                v_loss += loss_fn(logits, masks).item() * images.size(0); v_seen += images.size(0)
                accs.append(pixel_accuracy(logits, masks))
                pred = (torch.sigmoid(logits) > 0.5).float()
                inter = (pred * masks).sum(dim=(1, 2, 3))
                denom = pred.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3))
                dices += ((2 * inter + 1.0) / (denom + 1.0)).cpu().tolist()
        val_loss, val_acc, val_dice = v_loss / v_seen, float(np.mean(accs)), float(np.mean(dices))
        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc); history["val_dice"].append(val_dice)

        mark = ""
        if val_dice > best_score + 1e-4:
            best_score, best_epoch, waited = val_dice, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, f"/content/best_{name}.pt")
            mark = "  <- сохранили"
        else:
            waited += 1
            mark = f"  (без улучшения {waited}/{patience})"
        print(f"эпоха {epoch:2d}/{epochs}  train {train_loss:.4f}  val {val_loss:.4f}  "
              f"пикс.точность {val_acc:.4f}  Dice {val_dice:.4f}{mark}")
        if waited >= patience:
            print(f"ранняя остановка. Лучшая эпоха - {best_epoch}.")
            break
    model.load_state_dict(best_state)
    minutes = (time.time() - t0) / 60
    row = {"model": name, "loss": loss_fn.__class__.__name__, "epochs_run": len(history["val_dice"]),
           "best_epoch": best_epoch, "lr": lr, "img_size": IMG_SIZE, "seed": SEED,
           "best_val_dice": round(best_score, 4),
           "pixel_acc": round(history["val_acc"][best_epoch - 1], 4), "minutes": round(minutes, 1)}
    path = "/content/experiments.csv"
    pd.DataFrame([row]).to_csv(path, mode="a", header=not os.path.exists(path), index=False)
    print(f"лучший Dice: {best_score:.4f} (эпоха {best_epoch}) | {minutes:.1f} мин | записано в experiments.csv")
    return history, best_score


bce_model = UNet().to(device)
hist_bce, dice_bce = fit(bce_model, "unet_bce", nn.BCEWithLogitsLoss())


In [ ]:
# --- Первый результат: смотрим на числа и на картинки (инфраструктура) ---
final_acc = hist_bce["val_acc"][-1]
print(f"попиксельная точность на валидации: {final_acc:.4f}")
print(f"то есть {final_acc:.1%} пикселей предсказаны верно")
print()

# а сколько наберет модель, которая просто говорит «таблиц нет нигде»?
empty_correct, total_px = 0, 0
for _, masks in val_loader:
    empty_correct += float((masks == 0).sum()); total_px += masks.numel()
print(f"а модель, отвечающая «таблиц нет нигде», набрала бы: {empty_correct / total_px:.4f}")
print()

bce_model.eval()
images, masks = next(iter(val_loader))
with torch.no_grad():
    preds = (torch.sigmoid(bce_model(images.to(device))) > 0.5).float().cpu()
show = [i for i in range(len(masks)) if masks[i].any()][:3] or list(range(3))
fig, axes = plt.subplots(3, len(show), figsize=(4 * len(show), 11))
axes = np.atleast_2d(axes)
for j, i in enumerate(show):
    axes[0, j].imshow(images[i, 0], cmap="gray"); axes[0, j].axis("off"); axes[0, j].set_title("страница", fontsize=9)
    axes[1, j].imshow(masks[i, 0], cmap="gray"); axes[1, j].axis("off"); axes[1, j].set_title("эталон", fontsize=9)
    axes[2, j].imshow(preds[i, 0], cmap="gray"); axes[2, j].axis("off"); axes[2, j].set_title("предсказание", fontsize=9)
plt.tight_layout(); plt.show()
print("Сравните числа выше с тем, что видите на картинках. Сходится?")


In [ ]:
# --- ✅ Самопроверка Шага 2 ---
assert os.path.exists("/content/best_unet_bce.pt"), "Контрольная точка не сохранена."
assert probe_out.shape[1] == 1, "На выходе должен быть один канал: таблица или нет."
if len(hist_bce["train_loss"]) > 1:
    assert hist_bce["train_loss"][-1] < hist_bce["train_loss"][0], "Потеря не снизилась - сеть не учится."
baseline_acc = empty_correct / total_px
print(f"✅ Шаг 2 пройден: U-Net обучена, попиксельная точность {final_acc:.4f}.")
print(f"Пустая маска дала бы {baseline_acc:.4f} — разница всего {final_acc - baseline_acc:+.4f}.")
print()
print("Вот и главный вопрос этого шага: можно ли по такой метрике вообще судить о качестве?")
print("Ответ — на следующем шаге.")


### ✍️ Отчетная ячейка Шага 2
1) **Зачем U-Net пропускные связи** и что было бы без них: ...
2) **Попиксельная точность моей модели и точность «пустой маски»**. Насколько они различаются и о чем это говорит: ...
3) **Что я вижу на картинках** — находит ли модель таблицы или предпочитает молчать: ...

## Шаг 3. Честные метрики масок: IoU и Dice

Попиксельная точность обманула нас ровно так же, как доля верных ответов в Модуле 2: фон подавляюще велик, и модель получает высокий процент, ничего не найдя.

Нужна мера, которая сравнивает **области**, а не пиксели поодиночке. Таких две, и обе считаются из трех чисел: площадь эталона, площадь предсказания и площадь их пересечения.

In [ ]:
# --- TODO 3.1: посчитайте IoU и Dice ---
# Обе метрики считаются по одной паре масок за раз, на бинарных массивах из 0 и 1.
#   IoU  = пересечение / объединение
#   Dice = удвоенное пересечение / сумма площадей
# ВАЖНО про сглаживание: у нас две трети страниц без таблиц. Если эталон и предсказание
# оба пусты, обе формулы дадут 0/0. Добавка smooth в числитель и знаменатель это лечит:
# пустое против пустого честно дает 1.0, а на обычных масках добавка неразличима.

def iou_score(pred, target, smooth=1.0):
    """pred и target — бинарные тензоры одинаковой формы."""
    intersection = ...
    union = ...
    return ...

def dice_score(pred, target, smooth=1.0):
    intersection = ...
    total = ...
    return ...

# проверка на игрушечном примере: эталон 100 пикселей, предсказание 120, пересечение 80
a = torch.zeros(30, 30); a[0:10, 0:10] = 1                # эталон: 10x10 = 100 пикселей
b = torch.zeros(30, 30); b[2:12, 0:12] = 1                # предсказание: 10x12 = 120, пересечение 80
iou_demo, dice_demo = iou_score(b, a), dice_score(b, a)
if iou_demo is Ellipsis or dice_demo is Ellipsis:
    raise NotImplementedError("Заполните TODO 3.1: формулы IoU и Dice.")
print(f"игрушечный пример: IoU = {float(iou_demo):.3f}, Dice = {float(dice_demo):.3f}")
print(f"проверка связи 2*IoU/(1+IoU) = {2 * float(iou_demo) / (1 + float(iou_demo)):.3f}")


In [ ]:
# --- Считаем метрики модели по всей валидации (инфраструктура) ---
@torch.no_grad()
def evaluate_masks(model, loader, threshold=0.5):
    """Метрики считаются ПО КАЖДОЙ СТРАНИЦЕ и усредняются — как macro в Модуле 2."""
    model.eval()
    ious, dices, has_table = [], [], []
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        preds = (torch.sigmoid(model(images)) > threshold).float()
        for k in range(images.size(0)):
            ious.append(float(iou_score(preds[k], masks[k])))
            dices.append(float(dice_score(preds[k], masks[k])))
            has_table.append(bool(masks[k].any()))
    return np.array(ious), np.array(dices), np.array(has_table)

ious, dices, has_table = evaluate_masks(bce_model, val_loader)
print(f"по всем страницам:          IoU {ious.mean():.3f} | Dice {dices.mean():.3f}")
print(f"только там, где таблица есть: IoU {ious[has_table].mean():.3f} | "
      f"Dice {dices[has_table].mean():.3f}   ({has_table.sum()} страниц)")
print(f"там, где таблиц нет:          IoU {ious[~has_table].mean():.3f} | "
      f"Dice {dices[~has_table].mean():.3f}   ({(~has_table).sum()} страниц)")
print()
print("Обратите внимание: пустые страницы дают почти единицу и тянут среднее вверх.")
print("Именно поэтому в отчете всегда указывают, по каким страницам считалась метрика.")


In [ ]:
# --- Подбор порога по валидации (инфраструктура) ---
thresholds = np.arange(0.1, 0.95, 0.05)
scores_all, scores_with = [], []
for t in thresholds:
    _, d, ht = evaluate_masks(bce_model, val_loader, threshold=float(t))
    scores_all.append(d.mean())                                   # по ВСЕМ страницам
    scores_with.append(d[ht].mean() if ht.any() else 0.0)         # только там, где таблица есть
# порог выбираем по всем страницам: иначе критерий поощряет закрашивать побольше
# и не штрафует за ложные срабатывания на двух третях страниц без таблиц
best_t = float(thresholds[int(np.argmax(scores_all))])
scores = scores_all

plt.figure(figsize=(7, 3))
plt.plot(thresholds, scores_all, marker="o", color="#1F6FB2", label="по всем страницам")
plt.plot(thresholds, scores_with, marker="s", color="#B8860B", ls="--", label="только там, где таблица есть")
plt.axvline(best_t, color="#3C8C5A", ls="--", label=f"лучший порог {best_t:.2f}")
plt.axvline(0.5, color="#C0504D", ls=":", label="порог по умолчанию 0.5")
plt.xlabel("порог бинаризации"); plt.ylabel("Dice"); plt.legend()
plt.title("Порог — это настройка, а не константа"); plt.grid(alpha=0.3); plt.show()
print(f"лучший порог по всем страницам: {best_t:.2f} (Dice {max(scores_all):.3f})")
print(f"если бы подбирали только по страницам с таблицами, вышло бы "
      f"{float(thresholds[int(np.argmax(scores_with))]):.2f} - и модель закрашивала бы лишнее")
print()
print("Почему так. Две трети страниц у нас без таблиц. Критерий, который смотрит только")
print("на страницы с таблицами, ничего не теряет от ложных срабатываний на остальных")
print("и уводит порог вниз. Считать надо по всей валидации.")


In [ ]:
# --- ✅ Самопроверка Шага 3 ---
assert abs(float(iou_demo) - 0.571) < 0.02, \
    f"На игрушечном примере IoU должен быть около 0.571, получено {float(iou_demo):.3f}"
assert abs(float(dice_demo) - 0.727) < 0.02, \
    f"На игрушечном примере Dice должен быть около 0.727, получено {float(dice_demo):.3f}"
assert abs(2 * float(iou_demo) / (1 + float(iou_demo)) - float(dice_demo)) < 0.02, \
    "Связь Dice = 2*IoU/(1+IoU) не выполняется — проверьте формулы."

empty_pred = torch.zeros(10, 10); empty_true = torch.zeros(10, 10)
assert float(dice_score(empty_pred, empty_true)) > 0.99, \
    "Пустое против пустого должно давать 1.0 — добавьте сглаживание, иначе получите деление на ноль."

print(f"✅ Шаг 3 пройден: формулы верны, сглаживание работает.")
print(f"Dice на страницах с таблицами: {dices[has_table].mean():.3f} — вот честное число.")
print(f"Попиксельная точность была {final_acc:.3f}. Разница между ними и есть цена неправильной метрики.")


### ✍️ Отчетная ячейка Шага 3
1) **Почему попиксельная точность нам не подошла** — объясните на своих числах: ...
2) **Три числа Dice** (по всем страницам, только с таблицами, только без таблиц). Какое из них честнее показывать заказчику и почему: ...
3) **Что дал подбор порога** и почему 0.5 не является правильным ответом по умолчанию: ...

## Шаг 4. Dice Loss: учим тому, что меряем

Мы измерили беду честно. Теперь разберемся, откуда она берется.

Кросс-энтропия усредняет ошибку по всем пикселям поровну. На нашей странице таблица занимает около десятой части площади, а на двух третях страниц ее нет вовсе. Значит, огромная масса фоновых пикселей заглушает сигнал от объекта: сети выгоднее уверенно молчать, чем рисковать.

Лечение простое по идее: раз мы меряем совпадение областей, пусть модель его и оптимизирует. Превратим Dice из метрики в функцию потерь.

**Одна тонкость.** Метрику мы считали на бинарной маске — после порога. Но порог не дифференцируем, градиент через него не пройдет. Поэтому в функции потерь вместо бинарной маски используют сами вероятности: это называют мягким Dice.

In [ ]:
# --- Арифметика слепоты: почему BCE не видит таблицу (инфраструктура) ---
share = 0.10                      # доля площади таблицы на странице
p_bg = 0.02                       # сеть уверенно говорит «фон» везде
n_total = IMG_SIZE * IMG_SIZE
n_obj = int(share * n_total); n_bg = n_total - n_obj

bce_value = (n_bg * -np.log(1 - p_bg) + n_obj * -np.log(p_bg)) / n_total
soft_inter = n_obj * p_bg
dice_loss_value = 1 - (2 * soft_inter + 1) / (n_total * p_bg + n_obj + 1)

print("Предсказание «таблиц нет нигде» глазами двух функций потерь:")
print(f"  BCE:       {bce_value:.3f}   <- почти ноль, «все хорошо»")
print(f"  Dice Loss: {dice_loss_value:.3f}   <- почти максимум, тревога")
print()
print("Одно и то же бесполезное предсказание. Разница в том, что BCE усредняет по всем пикселям,")
print("а Dice устроен вокруг объекта: фон не входит в числитель вовсе.")


In [ ]:
# --- TODO 4.1: напишите Dice Loss ---
class DiceLoss(nn.Module):
    """Мягкий Dice: работает с вероятностями, а не с бинарной маской."""

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, target):
        # TODO 4.1 - четыре шага:
        #   1) переведите логиты в вероятности (какая функция это делает?);
        #   2) посчитайте «мягкое» пересечение: сумма произведения вероятностей и эталона;
        #   3) посчитайте сумму площадей: сумма вероятностей плюс сумма эталона;
        #   4) верните ПОТЕРЮ, а не метрику. Dice тем лучше, чем больше, а потерю
        #      оптимизатор минимизирует — подумайте, как одно превратить в другое.
        probs = ...
        intersection = ...
        total = ...
        dice = ...
        return ...


class ComboLoss(nn.Module):
    """Половина BCE, половина Dice: устойчивость первой плюс чувствительность второй."""

    def __init__(self, weight_bce=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.weight_bce = weight_bce

    def forward(self, logits, target):
        return self.weight_bce * self.bce(logits, target) + (1 - self.weight_bce) * self.dice(logits, target)


# проверка на понятных случаях
perfect = torch.full((1, 1, 8, 8), 10.0); perfect_target = torch.ones(1, 1, 8, 8)
awful = torch.full((1, 1, 8, 8), -10.0)
loss_fn_check = DiceLoss()
val_perfect, val_awful = loss_fn_check(perfect, perfect_target), loss_fn_check(awful, perfect_target)
if val_perfect is Ellipsis:
    raise NotImplementedError("Заполните TODO 4.1: мягкий Dice и превращение метрики в потерю.")
print(f"идеальное предсказание -> потеря {float(val_perfect):.4f} (ожидаем около 0)")
print(f"полностью неверное     -> потеря {float(val_awful):.4f} (ожидаем около 1)")


### 🔎 Третий способ: взвесить классы прямо в кросс-энтропии

Dice — не единственное лекарство. У `BCEWithLogitsLoss` есть параметр `pos_weight`: он умножает вклад пикселей объекта, компенсируя их малочисленность. Если таблица занимает десятую часть площади, разумная отправная точка — вес около отношения площадей фона к объекту.

Это дешевле Dice Loss по коду и часто работает не хуже. Ниже мы обучим и такую модель, чтобы сравнить все четыре варианта честно.

In [ ]:
# --- Считаем вес положительного класса (инфраструктура) ---
pos_pixels, total_pixels = 0.0, 0.0
for _, masks in train_loader:
    pos_pixels += float(masks.sum()); total_pixels += masks.numel()
pos_weight_value = (total_pixels - pos_pixels) / max(pos_pixels, 1.0)
print(f"пикселей таблиц: {pos_pixels / total_pixels:.1%} от всех")
print(f"отношение фона к объекту: {pos_weight_value:.1f} — таким и берем pos_weight")

weighted_bce = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight_value], device=device))


In [ ]:
# --- Обучаем еще две модели: чистый Dice и комбинация (инфраструктура) ---
dice_model = UNet().to(device)
hist_dice, dice_dice = fit(dice_model, "unet_dice", DiceLoss())

print()
combo_model = UNet().to(device)
hist_combo, dice_combo = fit(combo_model, "unet_combo", ComboLoss())

print()
wbce_model = UNet().to(device)
hist_wbce, dice_wbce = fit(wbce_model, "unet_weighted_bce", weighted_bce)


In [ ]:
# --- Сравнение трех функций потерь (инфраструктура) ---
rows = []
for label, m in [("BCE", bce_model), ("Dice", dice_model), ("BCE + Dice", combo_model),
                 ("взвешенная BCE", wbce_model)]:
    i, d, ht = evaluate_masks(m, val_loader, threshold=best_t)
    rows.append({"функция потерь": label,
                 "Dice (все страницы)": round(d.mean(), 3),
                 "Dice (есть таблица)": round(d[ht].mean(), 3),
                 "IoU (есть таблица)": round(i[ht].mean(), 3),
                 "ложные срабатывания на пустых": round(1 - d[~ht].mean(), 3)})
comparison = pd.DataFrame(rows)
display(comparison)
comparison.to_csv("/content/loss_comparison.csv", index=False)

best_row = comparison.loc[comparison["Dice (все страницы)"].idxmax()]
best_model_name = best_row["функция потерь"]
our_model = {"BCE": bce_model, "Dice": dice_model, "BCE + Dice": combo_model,
             "взвешенная BCE": wbce_model}[best_model_name]
print(f"лучшая функция потерь по Dice на всех страницах: {best_model_name}")
print()
print("Обратите внимание на две последние колонки. Модель может выигрывать на страницах")
print("с таблицами и при этом заливать лишним пустые страницы — в проде это одинаково плохо.")
print("Поэтому итоговый выбор делается по метрике на всей валидации.")


In [ ]:
# --- Смотрим глазами: что изменилось (инфраструктура) ---
images, masks = next(iter(val_loader))
idxs = [i for i in range(len(masks)) if masks[i].any()][:3] or list(range(3))
fig, axes = plt.subplots(6, len(idxs), figsize=(4 * len(idxs), 20))
axes = np.atleast_2d(axes)
titles = ["страница", "эталон", "BCE", "Dice", "BCE + Dice", "взвешенная BCE"]
with torch.no_grad():
    preds_all = [(torch.sigmoid(m(images.to(device))) > best_t).float().cpu()
                 for m in (bce_model, dice_model, combo_model, wbce_model)]
for j, i in enumerate(idxs):
    panels = [images[i, 0], masks[i, 0]] + [p[i, 0] for p in preds_all]
    for row, (panel, title) in enumerate(zip(panels, titles)):
        axes[row, j].imshow(panel, cmap="gray"); axes[row, j].axis("off")
        if j == 0:
            axes[row, j].set_ylabel(title)
        axes[row, j].set_title(title, fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# --- ✅ Самопроверка Шага 4 ---
assert float(val_perfect) < 0.05, f"На идеальном предсказании Dice Loss должен быть около нуля, получено {float(val_perfect):.3f}"
assert float(val_awful) > 0.9, f"На полностью неверном предсказании потеря должна быть около единицы, получено {float(val_awful):.3f}"
best_dice_with_table = comparison["Dice (все страницы)"].max()
bce_dice_with_table = comparison.loc[comparison["функция потерь"] == "BCE", "Dice (все страницы)"].iloc[0]
print(f"✅ Шаг 4 пройден. Dice на всех страницах: BCE {bce_dice_with_table:.3f} -> "
      f"лучшая {best_dice_with_table:.3f} ({best_dice_with_table - bce_dice_with_table:+.3f})")
if best_dice_with_table <= bce_dice_with_table + 0.01:
    print("Прирост небольшой — и это тоже результат. Посмотрите на картинки: возможно,")
    print("BCE справилась потому, что таблицы у нас довольно крупные. Опишите это в отчете.")


### ✍️ Отчетная ячейка Шага 4
1) **Почему BCE «не видит» таблицу** — перескажите арифметику своими словами: ...
2) **Что дала замена функции потерь** на моих данных (числа из таблицы сравнения): ...
3) **Что видно на картинках**: изменились ли границы масок, стало ли меньше пропусков: ...

### 🔎 Почему эта задача пограничная

Стоит сказать прямо: сегментация таблиц на сжатой странице — задача не из легких, и высоких чисел здесь ждать не приходится. Причина в самом признаке. Таблица отличается от абзаца текста **тонкими линейками и регулярной сеткой**, а это первое, что исчезает при уменьшении страницы: линия толщиной в один пиксель на исходном скане просто пропадает.

Мы работаем с размером 384, потому что на 768 обучение вчетверо дороже. Это осознанный компромисс, а не недосмотр — и именно такие компромиссы вы будете объяснять заказчику в Модуле 4.

### 🧪 Попробуйте сами (необязательно)

Каждый запуск автоматически пишет строку в `experiments.csv` — потом сравните.

- **Поднимите разрешение.** Поставьте `IMG_SIZE = 512` в ячейке настроек и перезапустите ноутбук с Шага 1. Обучение станет дольше, но линейки таблиц выживут при сжатии — это самый вероятный источник прироста.
- **Focal Loss** — уменьшает вклад «легких» пикселей, на которых модель и так уверена.
- **Другие веса в комбинации**: 0.3 и 0.7 вместо половины на половину.
- **Больше каналов**: `UNet(base=48)` или `base=64`. Модель тяжелее — окупится ли?
- **Другой `pos_weight`**: попробуйте вдвое больше и вдвое меньше расчетного.
- **Постобработка морфологией**: `cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)` заращивает дыры внутри предсказанной таблицы и часто поднимает Dice на пару пунктов без всякого дообучения.
- **Сегментация другого класса**: замените `TABLE_VALUE` на 1 (заголовки). Они занимают около процента площади — там дисбаланс проявит себя в полную силу, и разница между функциями потерь будет куда заметнее.

## Шаг 4б. Предобученный энкодер: тот же прием, что в Модуле 2

Наша U-Net учится с нуля — все ее веса инициализированы случайно. Но в Модуле 2 мы уже видели, насколько сильно помогает **перенос обучения**: там замороженное тело дало 0.495, а разморозка верхнего блока подняла результат до 0.68.

Тот же прием работает и в сегментации. Идея простая: энкодер U-Net — это обычная сверточная сеть, сжимающая изображение. Значит, вместо случайных весов туда можно поставить готовую ResNet-18, обученную на ImageNet, а декодер оставить обучаемым с нуля.

Собирать это руками не нужно: библиотека `segmentation_models_pytorch` делает такую сборку одной строкой.

**Важно для чистоты сравнения:** обучаем с той же функцией потерь, что победила выше, и на тех же данных. Меняется ровно одно — откуда взялись начальные веса энкодера.

In [ ]:
# --- Установка библиотеки (инфраструктура) ---
import sys, subprocess
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'segmentation-models-pytorch==0.3.4'
])
import segmentation_models_pytorch as smp
print('segmentation_models_pytorch', smp.__version__)


In [ ]:
# --- TODO 4б.1: соберите U-Net с предобученным энкодером ---
# Подсказки по API:
#   smp.Unet(encoder_name=..., encoder_weights=..., in_channels=..., classes=...)
#   encoder_name — имя энкодера, например "resnet18";
#   encoder_weights — откуда брать веса: "imagenet" или None;
#   in_channels — сколько каналов у наших страниц (мы работаем с серыми сканами);
#   classes — сколько каналов на выходе (у нас бинарная задача).
# Библиотека сама подгонит первый сверточный слой под нужное число входных каналов.

def build_pretrained_unet():
    return ...

pretrained_model = build_pretrained_unet()
if pretrained_model is Ellipsis or pretrained_model is None:
    raise NotImplementedError("Заполните TODO 4б.1: соберите smp.Unet с предобученным энкодером.")
pretrained_model = pretrained_model.to(device)

check = pretrained_model(xb[:2].to(device))
assert check.shape == (2, 1, IMG_SIZE, IMG_SIZE), \
    f"Выход должен быть (2, 1, {IMG_SIZE}, {IMG_SIZE}), получено {tuple(check.shape)} - "\
    "проверьте in_channels и classes."
print(f"параметров: {sum(p.numel() for p in pretrained_model.parameters()) / 1e6:.2f} млн "
      f"(у нашей U-Net было {sum(p.numel() for p in model.parameters()) / 1e6:.2f} млн)")

best_loss_fn = {"BCE": nn.BCEWithLogitsLoss(), "Dice": DiceLoss(), "BCE + Dice": ComboLoss(),
                "взвешенная BCE": weighted_bce}[best_row["функция потерь"]]
print(f"обучаем с той же функцией потерь, что победила выше: {best_row['функция потерь']}")

hist_pre, dice_pre = fit(pretrained_model, "unet_pretrained", best_loss_fn, lr=3e-4)


In [ ]:
# --- Сравнение: с нуля против предобученного (инфраструктура) ---
model_scratch_ref = our_model
i_scratch, d_scratch, ht = evaluate_masks(our_model, val_loader, threshold=best_t)
i_pre, d_pre, _ = evaluate_masks(pretrained_model, val_loader, threshold=best_t)

transfer_table = pd.DataFrame([
    {"энкодер": "с нуля (наша U-Net)",
     "Dice (все страницы)": round(float(d_scratch.mean()), 3),
     "Dice (есть таблица)": round(float(d_scratch[ht].mean()), 3),
     "IoU (есть таблица)": round(float(i_scratch[ht].mean()), 3),
     "параметров, млн": round(sum(p.numel() for p in our_model.parameters()) / 1e6, 2)},
    {"энкодер": "ResNet-18 с ImageNet",
     "Dice (все страницы)": round(float(d_pre.mean()), 3),
     "Dice (есть таблица)": round(float(d_pre[ht].mean()), 3),
     "IoU (есть таблица)": round(float(i_pre[ht].mean()), 3),
     "параметров, млн": round(sum(p.numel() for p in pretrained_model.parameters()) / 1e6, 2)},
])
display(transfer_table)
transfer_table.to_csv("/content/encoder_comparison.csv", index=False)

delta_all = float(d_pre.mean() - d_scratch.mean())
delta = float(d_pre[ht].mean() - d_scratch[ht].mean())
print(f"разница по всем страницам: {delta_all:+.3f}")
print(f"разница на страницах с таблицами: {delta:+.3f}")
print()
print("Почему предобученный энкодер помогает даже на документах, хотя ImageNet — это фотографии:")
print("нижние слои учат универсальные вещи — края, углы, повторяющиеся структуры.")
print("Сетка линеек таблицы для них не сложнее, чем решетка забора.")

# Основной критерий — Dice по всем страницам: он учитывает и пустые страницы,
# поэтому не поощряет модель, которая просто закрашивает больше площади.
pretrained_wins = (float(d_pre.mean()), float(d_pre[ht].mean())) > (
    float(d_scratch.mean()), float(d_scratch[ht].mean())
)
if pretrained_wins:
    our_model = pretrained_model
    best_model_name = best_row["функция потерь"] + " + предобученный энкодер"
    print(f"\nдальше используем предобученную модель: {best_model_name}")
else:
    print("\nпредобучение не дало выигрыша по всем страницам — остаемся на прежней модели")


### Финальная настройка порога выбранной модели

Первый порог выше подбирался для стартовой BCE-модели. После сравнения функций потерь и энкодеров победитель изменился, поэтому переносить на него старый порог методологически неверно. Веса повторно не обучаются: мы только выбираем рабочую границу вероятности для **уже выбранной** модели.


In [ ]:
# --- Финальный порог для модели, которая попадет в артефакт ---
FINAL_THRESHOLDS = np.arange(0.10, 0.95, 0.05)
final_scores_all, final_scores_with = [], []
for threshold in FINAL_THRESHOLDS:
    _, dice_values, has_table = evaluate_masks(
        our_model, val_loader, threshold=float(threshold)
    )
    final_scores_all.append(float(dice_values.mean()))
    final_scores_with.append(
        float(dice_values[has_table].mean()) if has_table.any() else 0.0
    )

old_threshold = float(best_t)
best_t = float(FINAL_THRESHOLDS[int(np.argmax(final_scores_all))])

plt.figure(figsize=(7, 3))
plt.plot(FINAL_THRESHOLDS, final_scores_all, marker="o", label="все страницы")
plt.plot(FINAL_THRESHOLDS, final_scores_with, marker="s", ls="--",
         label="страницы с таблицами")
plt.axvline(best_t, ls="--", label=f"финальный порог {best_t:.2f}")
plt.xlabel("порог бинаризации")
plt.ylabel("Dice")
plt.title("Порог выбран именно для финальной модели")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"порог стартовой BCE-модели: {old_threshold:.2f}")
print(f"порог финальной модели: {best_t:.2f}")
print(f"Dice по всем страницам при финальном пороге: {max(final_scores_all):.3f}")


In [ ]:
# --- Смотрим глазами: два варианта рядом (инфраструктура) ---
images, masks = next(iter(val_loader))
idxs = [i for i in range(len(masks)) if masks[i].any()][:3] or list(range(3))
with torch.no_grad():
    p_scratch = (torch.sigmoid(model_scratch_ref(images.to(device))) > best_t).float().cpu()
    p_pre = (torch.sigmoid(pretrained_model(images.to(device))) > best_t).float().cpu()

fig, axes = plt.subplots(4, len(idxs), figsize=(4 * len(idxs), 14))
axes = np.atleast_2d(axes)
titles = ["страница", "эталон", "с нуля", "предобученный энкодер"]
for j, i in enumerate(idxs):
    for row, panel in enumerate([images[i, 0], masks[i, 0], p_scratch[i, 0], p_pre[i, 0]]):
        axes[row, j].imshow(panel, cmap="gray"); axes[row, j].axis("off")
        axes[row, j].set_title(titles[row], fontsize=9)
plt.tight_layout(); plt.savefig("/content/encoder_comparison.png", dpi=130); plt.show()


### ✍️ Отчетная ячейка Шага 4б
1) **Что дал предобученный энкодер** — числа из таблицы и что видно на картинках: ...
2) **Почему признаки, выученные на фотографиях предметов, помогают на сканах документов** (подумайте, что учат нижние слои сверточной сети): ...
3) **Чем пришлось заплатить** — сравните число параметров и время обучения: ...

In [ ]:
# --- 🧪 Место для ваших экспериментов (необязательная ячейка) ---
# Пример 1: морфологическое закрытие дыр в предсказании — без дообучения.
#
# kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
# ious_m, dices_m = [], []
# our = bce_model.eval()
# with torch.no_grad():
#     for images, masks in val_loader:
#         probs = torch.sigmoid(our(images.to(device))).cpu().numpy()
#         for k in range(len(probs)):
#             raw = (probs[k, 0] > best_t).astype(np.uint8)
#             closed = cv2.morphologyEx(raw, cv2.MORPH_CLOSE, kernel)
#             t = masks[k, 0]
#             dices_m.append(float(dice_score(torch.from_numpy(closed).float(), t)))
# print("Dice после морфологии:", np.mean(dices_m))
#
# Пример 2: Focal Loss
#
# class FocalLoss(nn.Module):
#     def __init__(self, alpha=0.75, gamma=2.0):
#         super().__init__(); self.alpha, self.gamma = alpha, gamma
#     def forward(self, logits, target):
#         p = torch.sigmoid(logits)
#         ce = nn.functional.binary_cross_entropy_with_logits(logits, target, reduction="none")
#         p_t = p * target + (1 - p) * (1 - target)
#         alpha_t = self.alpha * target + (1 - self.alpha) * (1 - target)
#         return (alpha_t * (1 - p_t) ** self.gamma * ce).mean()
#
# focal_model = UNet().to(device)
# hist_focal, dice_focal = fit(focal_model, "unet_focal", FocalLoss())
print("Ячейка для необязательных экспериментов — раскомментируйте и пробуйте.")


## Шаг 5. Segment Anything: универсал без дообучения

Мы обучили специалиста: своя U-Net знает один класс — таблицы — и работает автономно. На другом полюсе живут фундаментальные модели: Segment Anything обучен на 1.1 миллиарда масок и выделяет что угодно по подсказке, **не видев ни одной нашей страницы**.

Задание ознакомительное: дообучать SAM не нужно. Задача — применить его через официальную библиотеку, сравнить с собственной моделью по IoU и сделать честный вывод, кто в чем сильнее.

Ключевое ограничение, которое стоит держать в голове с самого начала: SAM возвращает маску **без имени класса**. Он отвечает «вот объект», но не знает, таблица это или блок текста.

In [ ]:
# --- Установка SAM и загрузка весов (инфраструктура, 2–3 минуты) ---
import sys, subprocess, urllib.request
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/facebookresearch/segment-anything.git'
])

SAM_CHECKPOINT = '/content/sam_vit_b_01ec64.pth'
if not os.path.exists(SAM_CHECKPOINT):
    urllib.request.urlretrieve(
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        SAM_CHECKPOINT,
    )

from segment_anything import sam_model_registry, SamPredictor
if not os.path.exists(SAM_CHECKPOINT):
    raise FileNotFoundError('Не скачались веса SAM. Проверьте интернет и повторите ячейку.')

sam = sam_model_registry['vit_b'](checkpoint=SAM_CHECKPOINT).to(device)
predictor = SamPredictor(sam)
print(f'SAM загружен: {os.path.getsize(SAM_CHECKPOINT) / 1e6:.0f} МБ весов')
print('устройство:', device)


In [ ]:
# --- TODO 5.1: подскажите SAM, где искать таблицу ---
# Берем страницы, где таблица есть, и даем SAM подсказку.
# Виды подсказок: точка внутри объекта (positive), точка на фоне (negative), рамка.
# Мы поступим честно: рамку строим по ЭТАЛОННОЙ маске — то есть даем SAM максимальную фору.
# Так сравнение будет в его пользу, и вывод получится осторожным.

sam_pages = [n for n in val_pages
             if (cv2.imread(str(PAGES / "masks" / f"{n}.png"), cv2.IMREAD_GRAYSCALE) == TABLE_VALUE).any()][:6]
print(f"страниц для сравнения: {len(sam_pages)}")

def bbox_from_mask(mask):
    """Рамка вокруг всех ненулевых пикселей маски: (x0, y0, x1, y1)."""
    rows, cols = np.where(mask > 0)
    return np.array([cols.min(), rows.min(), cols.max(), rows.max()])

def sam_predict(image_rgb, box):
    """Возвращает бинарную маску от SAM по подсказке-рамке."""
    predictor.set_image(image_rgb)              # тяжелый кодировщик: один раз на страницу
    # TODO 5.1 — вызовите predictor.predict.
    #   Аргументы: box=box[None, :], multimask_output=True.
    #   Метод возвращает три вещи: маски, оценки качества и логиты.
    #   SAM отдает ТРИ маски-кандидата — выберите ту, у которой предсказанная оценка выше,
    #   и верните ее как булев массив.
    masks, scores, _ = ...
    best = ...
    return masks[best]

probe_name = sam_pages[0]
probe_img = cv2.cvtColor(cv2.imread(str(PAGES / "images" / f"{probe_name}.jpg")), cv2.COLOR_BGR2RGB)
probe_mask = (cv2.imread(str(PAGES / "masks" / f"{probe_name}.png"), cv2.IMREAD_GRAYSCALE) == TABLE_VALUE)
probe_result = sam_predict(probe_img, bbox_from_mask(probe_mask))
if probe_result is Ellipsis or probe_result is None:
    raise NotImplementedError("Заполните TODO 5.1: вызов predictor.predict и выбор лучшего кандидата.")
print(f"SAM вернул маску формы {probe_result.shape}, покрытие {probe_result.mean():.1%} страницы")


In [ ]:
# --- Сравнение SAM и нашей U-Net по IoU (инфраструктура) ---
# our_model уже выбрана на Шаге 4б: это лучший вариант из всех обученных
our_model.eval()

rows, panels = [], []
t_sam = time.time()
for name in sam_pages:
    page = cv2.imread(str(PAGES / "images" / f"{name}.jpg"), cv2.IMREAD_GRAYSCALE)
    truth = (cv2.imread(str(PAGES / "masks" / f"{name}.png"), cv2.IMREAD_GRAYSCALE) == TABLE_VALUE)

    small = cv2.resize(page, (IMG_SIZE, IMG_SIZE))
    tensor = torch.from_numpy(small).float().unsqueeze(0).unsqueeze(0).to(device) / 255.0
    with torch.no_grad():
        pred_small = (torch.sigmoid(our_model(tensor)) > best_t).float().cpu().numpy()[0, 0]
    ours = cv2.resize(pred_small, truth.shape[::-1], interpolation=cv2.INTER_NEAREST) > 0.5

    sam_mask = sam_predict(cv2.cvtColor(page, cv2.COLOR_GRAY2RGB), bbox_from_mask(truth))

    to_t = lambda a: torch.from_numpy(a.astype(np.float32))
    rows.append({"страница": name,
                 "IoU нашей U-Net": round(float(iou_score(to_t(ours), to_t(truth))), 3),
                 "IoU SAM (с рамкой-подсказкой)": round(float(iou_score(to_t(sam_mask), to_t(truth))), 3)})
    panels.append((page, truth, ours, sam_mask))
sam_time = (time.time() - t_sam) / len(sam_pages)

sam_table = pd.DataFrame(rows)
display(sam_table)
sam_table.to_csv("/content/sam_comparison.csv", index=False)
print(f"в среднем: наша U-Net {sam_table['IoU нашей U-Net'].mean():.3f} | "
      f"SAM {sam_table['IoU SAM (с рамкой-подсказкой)'].mean():.3f}")
print(f"SAM тратит около {sam_time:.1f} с на страницу (включая кодирование изображения)")


In [ ]:
# --- Смотрим глазами (инфраструктура) ---
fig, axes = plt.subplots(4, len(panels), figsize=(3.3 * len(panels), 13))
axes = np.atleast_2d(axes)
titles = ["страница", "эталон", "наша U-Net", "SAM по рамке"]
for j, (page, truth, ours, sam_mask) in enumerate(panels):
    for row, panel in enumerate([page, truth, ours, sam_mask]):
        axes[row, j].imshow(panel, cmap="gray"); axes[row, j].axis("off")
        axes[row, j].set_title(titles[row], fontsize=8)
plt.tight_layout(); plt.savefig("/content/sam_vs_unet.png", dpi=130); plt.show()


In [ ]:
# --- ✅ Самопроверка Шага 5 ---
assert len(sam_table) >= 3, "Для сравнения нужно хотя бы три страницы."
assert sam_table["IoU SAM (с рамкой-подсказкой)"].mean() > 0.1, \
    "SAM выдал почти пустые маски — проверьте, что рамка передана в формате (x0, y0, x1, y1)."
print("✅ Шаг 5 пройден.")
print(f"наша U-Net: {sam_table['IoU нашей U-Net'].mean():.3f} | SAM с подсказкой: "
      f"{sam_table['IoU SAM (с рамкой-подсказкой)'].mean():.3f}")
print()
print("Помните об условиях сравнения: SAM получил рамку, построенную по ЭТАЛОННОЙ маске,")
print("то есть человек уже указал ему, где искать. Наша U-Net работала сама, без подсказок.")


### ✍️ Отчетная ячейка Шага 5

Сравните две модели по осям, а не по одному числу:

| Ось | Наша U-Net | SAM |
|---|---|---|
| Знание класса | ✍️ | ✍️ |
| Нужна ли разметка для запуска | ✍️ | ✍️ |
| Автономность (работает ли без человека) | ✍️ | ✍️ |
| Скорость на страницу | ✍️ | ✍️ |
| Качество границ | ✍️ | ✍️ |

1) **Кто победил и в чем именно** (учтите, что сравнение было в пользу SAM: он получил подсказку): ...
2) **Сценарий, где я выберу SAM, и сценарий, где выберу свою модель**: ...
3) **Паттерн «SAM размечает — своя модель служит»**: как бы вы использовали SAM, чтобы удешевить разметку следующих пятисот страниц: ...

## Шаг 6. От маски к вырезанной таблице

Маска — внутренний формат конвейера. Заказчику нужен фрагмент: вырезанная картинка с таблицей, которую можно отдать бухгалтеру или системе распознавания.

Между маской и фрагментом четыре шага, и каждый умеет тихо все испортить:

1. **Чистка.** Сырая маска рваная: мелкие пятна на фоне превратят рамку в размер всей страницы.
2. **Рамка.** Минимальный прямоугольник вокруг оставшейся области.
3. **Поля и клипинг.** Немного контекста вокруг таблицы, но не за границами страницы.
4. **Пересчет координат.** Модель работала с уменьшенной страницей, а вырезать надо из оригинала.

In [ ]:
# --- TODO 6.1: маска -> список рамок ---
def mask_to_boxes(mask, min_area_share=0.02, margin=0.05, original_shape=None, min_side=24):
    """Превращает бинарную маску в список рамок (x0, y0, x1, y1) в координатах оригинала.

    mask — бинарный массив в разрешении модели;
    min_area_share — минимальная доля площади кадра, ниже которой компонента считается шумом;
    margin — поле вокруг рамки в долях ее размера;
    original_shape — (высота, ширина) исходной страницы для пересчета координат.
    """
    mask_u8 = (mask > 0).astype(np.uint8)
    h, w = mask_u8.shape

    # TODO 6.1a — найдите компоненты связности.
    #   Подсказка: cv2.connectedComponentsWithStats(mask_u8, connectivity=8) возвращает
    #   число компонент, карту меток и статистику stats, где для каждой компоненты есть
    #   x, y, ширина, высота и площадь (индексы cv2.CC_STAT_*).
    #   Компонента 0 — это фон, ее всегда пропускаем.
    components = ...
    if components is Ellipsis:
        raise NotImplementedError("Заполните TODO 6.1a: вызов cv2.connectedComponentsWithStats.")
    n_labels, labels, stats, _ = components

    boxes = []
    for i in range(1, n_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        # TODO 6.1b - отбросьте шум: пропустите компоненту, если ее площадь меньше
        #   min_area_share от площади всего кадра. Почему порог в долях, а не в пикселях?
        min_area = ...
        if min_area is Ellipsis:
            raise NotImplementedError("Заполните TODO 6.1b: порог площади в долях кадра.")
        if area < min_area:
            continue
        x, y = stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP]
        bw, bh = stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]
        x0, y0, x1, y1 = x, y, x + bw, y + bh

        if original_shape is not None:                 # пересчет в координаты оригинала
            scale_x, scale_y = original_shape[1] / w, original_shape[0] / h
            x0, x1 = int(x0 * scale_x), int(x1 * scale_x)
            y0, y1 = int(y0 * scale_y), int(y1 * scale_y)
            limit_h, limit_w = original_shape
        else:
            limit_h, limit_w = h, w

        # TODO 6.1c - добавьте поля и прижмите рамку к границам страницы.
        #   Поле: margin от ширины и высоты рамки соответственно.
        #   Клипинг: координаты не должны выйти за пределы страницы. Что произойдет,
        #   если этого не сделать и x0 окажется отрицательным? Вспомните, как NumPy
        #   понимает отрицательные индексы.
        if (x1 - x0) < min_side or (y1 - y0) < min_side:     # обрывки текста и полоски отсеиваем
            continue
        pad_x, pad_y = int(margin * (x1 - x0)), int(margin * (y1 - y0))
        x0, y0 = ..., ...
        x1, y1 = ..., ...
        if any(v is Ellipsis for v in (x0, y0, x1, y1)):
            raise NotImplementedError("Заполните TODO 6.1c: поля и клипинг по границам страницы.")
        boxes.append((x0, y0, x1, y1))
    return boxes


# проверка на игрушечной маске: большой прямоугольник и одно шумовое пятно
demo_mask = np.zeros((100, 100), np.uint8)
demo_mask[20:70, 15:85] = 1          # «таблица»
demo_mask[2:5, 95:98] = 1            # шум в углу
demo_boxes = mask_to_boxes(demo_mask, min_area_share=0.02, margin=0.0)
if demo_boxes is Ellipsis or demo_boxes is None:
    raise NotImplementedError("Заполните TODO 6.1: компоненты связности, фильтр шума, поля и клипинг.")
print(f"найдено рамок: {len(demo_boxes)} (ожидаем 1: шумовое пятно должно отсеяться)")
print("рамка:", demo_boxes)


In [ ]:
# --- Вырезаем таблицы из валидационных страниц (инфраструктура) ---
CROPS_DIR = "/content/crops"
shutil.rmtree(CROPS_DIR, ignore_errors=True); os.makedirs(CROPS_DIR)

crop_records = []
for name in val_pages:
    page = cv2.imread(str(PAGES / "images" / f"{name}.jpg"), cv2.IMREAD_GRAYSCALE)
    small = cv2.resize(page, (IMG_SIZE, IMG_SIZE))
    tensor = torch.from_numpy(small).float().unsqueeze(0).unsqueeze(0).to(device) / 255.0
    with torch.no_grad():
        pred = (torch.sigmoid(our_model(tensor)) > best_t).float().cpu().numpy()[0, 0]
    boxes = mask_to_boxes(pred, original_shape=page.shape)
    for k, (x0, y0, x1, y1) in enumerate(boxes):
        crop = page[y0:y1, x0:x1].copy()          # .copy(): срез NumPy - это окно в исходный массив
        if crop.size == 0:
            continue
        out_name = f"{name}_table{k}.png"
        cv2.imwrite(f"{CROPS_DIR}/{out_name}", crop)
        crop_records.append({"page": name, "crop": out_name,
                             "x0": x0, "y0": y0, "x1": x1, "y1": y1,
                             "width": x1 - x0, "height": y1 - y0})

crops_df = pd.DataFrame(crop_records)
crops_df.to_csv("/content/crops.csv", index=False)
print(f"вырезано фрагментов: {len(crops_df)} со {crops_df['page'].nunique()} страниц")
print(f"страниц без единой найденной таблицы: {len(val_pages) - crops_df['page'].nunique()}")
if len(crops_df):
    print(f"средний размер фрагмента: {crops_df['width'].mean():.0f} x {crops_df['height'].mean():.0f}")


In [ ]:
# --- Контрольная сетка: страница с рамкой и сам фрагмент (инфраструктура) ---
sample_crops = crops_df.sample(min(4, len(crops_df)), random_state=SEED) if len(crops_df) else crops_df
fig, axes = plt.subplots(2, max(len(sample_crops), 1), figsize=(3.6 * max(len(sample_crops), 1), 9))
axes = np.atleast_2d(axes)
for j, (_, row) in enumerate(sample_crops.iterrows()):
    page = cv2.cvtColor(cv2.imread(str(PAGES / "images" / f"{row['page']}.jpg")), cv2.COLOR_BGR2RGB)
    marked = page.copy()
    cv2.rectangle(marked, (row["x0"], row["y0"]), (row["x1"], row["y1"]), (200, 40, 40), 4)
    axes[0, j].imshow(marked); axes[0, j].axis("off"); axes[0, j].set_title(row["page"], fontsize=9)
    axes[1, j].imshow(cv2.imread(f"{CROPS_DIR}/{row['crop']}", cv2.IMREAD_GRAYSCALE), cmap="gray")
    axes[1, j].axis("off"); axes[1, j].set_title(f"{row['width']}x{row['height']}", fontsize=9)
plt.tight_layout(); plt.savefig("/content/crops_preview.png", dpi=130); plt.show()
print("Так выглядит то, что получит заказчик: вырезанная таблица вместо целой страницы.")


In [ ]:
# --- ✅ Самопроверка Шага 6 ---
assert len(demo_boxes) == 1, \
    f"На игрушечной маске должна остаться одна рамка, получено {len(demo_boxes)} — проверьте фильтр по площади."
dx0, dy0, dx1, dy1 = demo_boxes[0]
assert dx0 >= 0 and dy0 >= 0, "Координаты рамки отрицательные — забыли клипинг."
assert abs(dx0 - 15) <= 2 and abs(dy0 - 20) <= 2, \
    f"Рамка должна обнимать прямоугольник (15, 20)—(85, 70), получено ({dx0}, {dy0})-({dx1}, {dy1})"

edge_mask = np.zeros((100, 100), np.uint8); edge_mask[0:30, 0:30] = 1      # объект в самом углу
edge_boxes = mask_to_boxes(edge_mask, margin=0.3)
assert all(x >= 0 and y >= 0 for x, y, _, _ in edge_boxes), \
    "У объекта в углу поле увело координаты в минус — клипинг не работает."

assert len(crops_df) > 0, "Не вырезано ни одного фрагмента — проверьте порог и фильтр площади."
print(f"✅ Шаг 6 пройден: фильтр шума работает, клипинг на месте, вырезано {len(crops_df)} фрагментов.")


### ✍️ Отчетная ячейка Шага 6
1) **Зачем чистить маску перед построением рамки** — что было бы с одним шумовым пятном в углу: ...
2) **Почему порог площади задан в долях кадра, а не в пикселях**: ...
3) **Что произойдет без клипинга**, если объект стоит у самого края страницы (вспомните отрицательные индексы): ...
4) **Оцените результат глазами**: сколько таблиц вырезано аккуратно, где рамка захватила лишнее или обрезала нужное: ...
5) **Настройки фильтра.** У функции два порога: минимальная доля площади и минимальная сторона рамки. Попробуйте сделать их строже и мягче — как меняется число вырезанных фрагментов и их качество: ...

## Шаг 7. Проверка на данных заказчика: собираем эстафету

До сих пор мы работали с размеченными страницами от подрядчика. Но у заказчика архив из первого задания — другие документы, другое качество сканирования, никакой разметки.

Здесь пригодится **ваш классификатор из ДЗ 2**: он отберет документы, в которых таблицы ожидаются по смыслу — счета и бюджеты, — и на них мы запустим U-Net. Проверить по эталону не получится (масок нет), поэтому оцениваем глазами и по косвенным признакам.

In [ ]:
# --- TODO 7.1: загрузите классификатор из ДЗ 2 по его контракту ---
metrics_dz2 = json.load(open(DZ2 / 'metrics.json', encoding='utf-8'))
config_path = DZ2 / 'model_config.json'
checkpoint_path = DZ2 / 'classifier.pt' if (DZ2 / 'classifier.pt').exists() else DZ2 / 'best_model.pt'
payload = torch.load(checkpoint_path, map_location='cpu')
checkpoint_config = payload.get('config', {}) if isinstance(payload, dict) else {}
state_dict = payload.get('state_dict', payload) if isinstance(payload, dict) else payload
if not isinstance(state_dict, dict):
    raise TypeError('Checkpoint ДЗ 2 должен содержать state_dict.')
state_dict = {str(key).removeprefix('module.'): value for key, value in state_dict.items()}

if config_path.exists():
    DZ2_MODEL_CONFIG = json.load(open(config_path, encoding='utf-8'))
    DZ2_CONFIG_SOURCE = 'model_config.json'
elif checkpoint_config:
    DZ2_MODEL_CONFIG = checkpoint_config
    DZ2_CONFIG_SOURCE = checkpoint_path.name + ':config'
else:
    # Обратная совместимость со старым архивом ДЗ 2.
    DZ2_MODEL_CONFIG = {
        'framework': 'torchvision',
        'architecture': 'resnet18',
        'classes': list(metrics_dz2['classes']),
        'preprocessing': {
            'image_size': int(metrics_dz2.get('img_size', 224)),
            'input_color': 'grayscale_triplicate',
            'resize_policy': 'square_resize',
            'resize_interpolation': 'linear',
            'mean': [0.485, 0.456, 0.406],
            'std': [0.229, 0.224, 0.225],
        },
    }
    DZ2_CONFIG_SOURCE = 'legacy inference'

DZ2_CLASSES = list(DZ2_MODEL_CONFIG.get('classes', metrics_dz2['classes']))
DZ2_ARCHITECTURE = str(DZ2_MODEL_CONFIG.get('architecture', 'resnet18')).lower()
DZ2_FRAMEWORK = str(DZ2_MODEL_CONFIG.get('framework', 'torchvision')).lower()
DZ2_PREPROCESSING = dict(DZ2_MODEL_CONFIG.get('preprocessing', {}))
CLS_SIZE = int(DZ2_PREPROCESSING.get('image_size', 224))
CLS_INPUT_COLOR = str(DZ2_PREPROCESSING.get('input_color', 'grayscale_triplicate')).lower()
CLS_RESIZE_POLICY = str(DZ2_PREPROCESSING.get('resize_policy', 'square_resize')).lower()
CLS_RESIZE_INTERPOLATION = str(DZ2_PREPROCESSING.get('resize_interpolation', 'linear')).lower()
CLS_MEAN = tuple(DZ2_PREPROCESSING.get('mean', (0.485, 0.456, 0.406)))
CLS_STD = tuple(DZ2_PREPROCESSING.get('std', (0.229, 0.224, 0.225)))


def build_dz2_classifier():
    # TODO 7.1a — восстановите архитектуру по DZ2_FRAMEWORK и DZ2_ARCHITECTURE,
    # замените голову на len(DZ2_CLASSES), загрузите state_dict и верните модель.
    model = ...
    ...
    return model


classifier = build_dz2_classifier()
if classifier is Ellipsis or classifier is None:
    raise NotImplementedError('Заполните TODO 7.1a: восстановите классификатор из артефакта.')
classifier = classifier.to(device).eval()
print('классы из ДЗ 2:', DZ2_CLASSES)
print('модель:', f'{DZ2_FRAMEWORK}/{DZ2_ARCHITECTURE}', '| config:', DZ2_CONFIG_SOURCE)


def classifier_interpolation():
    mapping = {
        'linear': cv2.INTER_LINEAR,
        'bilinear': cv2.INTER_LINEAR,
        'area': cv2.INTER_AREA,
        'nearest': cv2.INTER_NEAREST,
        'cubic': cv2.INTER_CUBIC,
    }
    if CLS_RESIZE_INTERPOLATION not in mapping:
        raise ValueError(f'Неизвестная интерполяция: {CLS_RESIZE_INTERPOLATION}')
    return mapping[CLS_RESIZE_INTERPOLATION]


def prepare_for_classifier(path):
    # TODO 7.1b — прочитайте изображение в цвете, указанном в CLS_INPUT_COLOR;
    # выполните resize_policy из контракта; переведите в [0,1], нормализуйте
    # CLS_MEAN/CLS_STD и верните tensor CxHxW.
    ...


labels_csv = pd.read_csv(DZ2 / 'clean_labels.csv')
sample_files = labels_csv['file'].sample(min(120, len(labels_csv)), random_state=SEED).tolist()
WANTED = [class_name for class_name in ('invoice', 'budget') if class_name in DZ2_CLASSES]
wanted_indices = [DZ2_CLASSES.index(class_name) for class_name in WANTED]
selected = []
with torch.no_grad():
    for relative_path in sample_files:
        path = ARCHIVE / relative_path
        if not path.exists():
            continue
        tensor = prepare_for_classifier(path)
        if tensor is Ellipsis or tensor is None:
            raise NotImplementedError('Заполните TODO 7.1b: предобработка должна вернуть tensor.')
        logits = classifier(tensor.unsqueeze(0).to(device))
        predicted_index = int(logits.argmax(1))
        if predicted_index in wanted_indices:
            selected.append((relative_path, DZ2_CLASSES[predicted_index]))
        if len(selected) >= 6:
            break

print(f'классификатор отобрал {len(selected)} документов типов {WANTED} из {len(sample_files)} просмотренных')
for relative_path, class_name in selected:
    print(f'   {relative_path} -> {class_name}')


In [ ]:
# --- Запускаем U-Net на сканах заказчика (инфраструктура) ---
if not selected:
    print("⚠️ Классификатор не отобрал ни одного счета или бюджета в этой выборке.")
    print("   Возьмите случайные страницы, чтобы шаг можно было выполнить.")
    selected = [(rel, "случайная страница") for rel in sample_files[:4]]

transfer_rows = []
fig, axes = plt.subplots(2, len(selected), figsize=(3.4 * len(selected), 8.5))
axes = np.atleast_2d(axes)
for j, (rel, cls) in enumerate(selected):
    page = cv2.imread(str(ARCHIVE / rel), cv2.IMREAD_GRAYSCALE)
    small = cv2.resize(page, (IMG_SIZE, IMG_SIZE))
    tensor = torch.from_numpy(small).float().unsqueeze(0).unsqueeze(0).to(device) / 255.0
    with torch.no_grad():
        prob = torch.sigmoid(our_model(tensor)).cpu().numpy()[0, 0]
    pred = prob > best_t
    boxes = mask_to_boxes(pred, original_shape=page.shape)
    transfer_rows.append({"file": rel, "класс по ДЗ 2": cls,
                          "доля площади под маской": round(float(pred.mean()), 4),
                          "найдено рамок": len(boxes)})
    axes[0, j].imshow(page, cmap="gray"); axes[0, j].axis("off"); axes[0, j].set_title(cls, fontsize=9)
    small_rgb = cv2.cvtColor(cv2.resize(page, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_GRAY2RGB)
    cnts, _ = cv2.findContours(pred.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(small_rgb, cnts, -1, (220, 40, 40), 3)
    axes[1, j].imshow(small_rgb); axes[1, j].axis("off")
    axes[1, j].set_title(f"рамок: {len(boxes)}", fontsize=9)
plt.tight_layout(); plt.savefig("/content/transfer_check.png", dpi=130); plt.show()

transfer_df = pd.DataFrame(transfer_rows)
display(transfer_df)
transfer_df.to_csv("/content/transfer_check.csv", index=False)

val_share = float(np.mean([(cv2.imread(str(PAGES / "masks" / f"{n}.png"), cv2.IMREAD_GRAYSCALE) == TABLE_VALUE).mean()
                           for n in val_pages]))
print(f"для сравнения: на размеченных страницах таблицы занимают {val_share:.1%} площади,")
print(f"а на сканах заказчика модель закрашивает {transfer_df['доля площади под маской'].mean():.1%}")


In [ ]:
# --- ✅ Самопроверка Шага 7 ---
assert len(transfer_df) >= 3, "Нужно проверить хотя бы три документа заказчика."
assert "доля площади под маской" in transfer_df.columns
print("✅ Шаг 7 пройден: эстафета собрана - классификатор из ДЗ 2 отобрал документы,")
print("   U-Net из ДЗ 3 нашла на них таблицы.")
print()
print("Оценивать качество по эталону здесь невозможно: масок для этих сканов не существует.")
print("Поэтому вывод делается глазами и по косвенным признакам — именно так и бывает,")
print("когда решение переносят на данные, которых модель не видела.")


### ✍️ Отчетная ячейка Шага 7
1) **Что показала проверка на данных заказчика**: находит ли модель таблицы, где ошибается: ...
2) **Как отличаются данные подрядчика и заказчика** (качество сканирования, макеты, типы документов) и что из этого объясняет разницу: ...
3) **Что бы вы сделали, чтобы улучшить перенос** — и во что это обошлось бы заказчику: ...

## Шаг 8. Собираем артефакты для ДЗ 4

In [ ]:
# --- Манифест результатов и контракт модели (инфраструктура) ---
final_i, final_d, final_ht = evaluate_masks(our_model, val_loader, threshold=best_t)

# Попиксельную точность также пересчитываем для финальной модели и ее порога,
# а не переносим число от стартовой BCE-модели.
final_correct, final_pixels = 0, 0
our_model.eval()
with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        predictions = (torch.sigmoid(our_model(images)) > best_t).float()
        final_correct += int((predictions == masks).sum().item())
        final_pixels += int(masks.numel())
final_acc = final_correct / max(final_pixels, 1)

# Явный контракт нужен следующему заданию. Раньше в архиве лежал только state_dict,
# но победителем могла стать одна из двух несовместимых архитектур.
DZ3_ARCHITECTURE = (
    "smp_unet_resnet18" if "предобученный" in best_model_name.lower()
    else "custom_unet"
)
DZ3_MODEL_CONFIG = {
    "schema_version": 2,
    "task": "binary_table_segmentation",
    "architecture": DZ3_ARCHITECTURE,
    "encoder_name": "resnet18" if DZ3_ARCHITECTURE == "smp_unet_resnet18" else None,
    "encoder_weights_during_training": (
        "imagenet" if DZ3_ARCHITECTURE == "smp_unet_resnet18" else None
    ),
    "base_channels": 32 if DZ3_ARCHITECTURE == "custom_unet" else None,
    "in_channels": 1,
    "classes": 1,
    "image_size": int(IMG_SIZE),
    "input_color": "grayscale",
    "input_scale": "0_1",
    "resize_interpolation": "linear",
    "normalization": None,
    "threshold": round(float(best_t), 4),
    "postprocessing": {
        "min_area_share": 0.02,
        "margin": 0.05,
        "min_side": 24,
        "connectivity": 8,
    },
    "recommended_document_types": ["invoice", "budget"],
    "state_dict_file": "best_unet.pt",
    "checkpoint_file": "table_extractor.pt",
    "selected_model_description": best_model_name,
}

RESULTS = {
    "seed": SEED,
    "device": device,
    "img_size": IMG_SIZE,
    "max_epochs": EPOCHS,
    "target_class": "table",
    "train_pages": len(train_pages),
    "val_pages": len(val_pages),
    "table_area_share": round(float(shares.mean()), 4),
    "empty_masks_share": round(float(empties / len(train_pages)), 4),
    "best_threshold": round(best_t, 2),
    "losses": {
        "bce": {"dice_all": round(float(dices.mean()), 4)},
        "comparison_table": comparison.to_dict(orient="records"),
    },
    "best_loss": best_model_name,
    "model_artifact": DZ3_MODEL_CONFIG,
    "encoder_comparison": transfer_table.to_dict(orient="records"),
    "pretrained_gain_all_pages": round(float(delta_all), 4),
    "pretrained_gain_pages_with_table": round(float(delta), 4),
    "final": {
        "dice_all_pages": round(float(final_d.mean()), 4),
        "dice_pages_with_table": round(float(final_d[final_ht].mean()), 4),
        "iou_pages_with_table": round(float(final_i[final_ht].mean()), 4),
        "pixel_accuracy": round(float(final_acc), 4),
    },
    "sam": {
        "our_iou": round(float(sam_table["IoU нашей U-Net"].mean()), 4),
        "sam_iou": round(float(sam_table["IoU SAM (с рамкой-подсказкой)"].mean()), 4),
        "sam_seconds_per_page": round(float(sam_time), 2),
    },
    "crops_extracted": int(len(crops_df)),
    "transfer_mask_share": round(float(transfer_df["доля площади под маской"].mean()), 4),
    "chosen_loss_reason": "заполните: какую функцию потерь рекомендуете и почему",
    "sam_verdict": "заполните: когда брать SAM, а когда свою модель — одной фразой",
    "transfer_verdict": "заполните: переносится ли модель на данные заказчика и что мешает",
}
with open("/content/metrics_dz3.json", "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, ensure_ascii=False, indent=2)

with open("/content/model_config.json", "w", encoding="utf-8") as f:
    json.dump(DZ3_MODEL_CONFIG, f, ensure_ascii=False, indent=2)

print("=== СВОДКА ВАШЕЙ РАБОТЫ ===")
for k, v in RESULTS.items():
    print(f"  {k}: {v}")
print()
print("контракт модели:", DZ3_MODEL_CONFIG)
print("⚠️ Три текстовых поля заполните руками — это часть отчета.")


In [ ]:
# --- Упаковка dz3_artifacts.zip (инфраструктура) ---
ART = "/content/dz3_artifacts"
shutil.rmtree(ART, ignore_errors=True)
os.makedirs(ART)

weights = {
    "BCE": "/content/best_unet_bce.pt",
    "Dice": "/content/best_unet_dice.pt",
    "BCE + Dice": "/content/best_unet_combo.pt",
    "взвешенная BCE": "/content/best_unet_weighted_bce.pt",
}
weight_path = (
    "/content/best_unet_pretrained.pt"
    if "предобученный" in best_model_name.lower()
    else weights[best_model_name]
)

# 1) Старое имя оставляем для обратной совместимости.
shutil.copy(weight_path, os.path.join(ART, "best_unet.pt"))

# 2) Новый checkpoint сам содержит и state_dict, и конфигурацию.
raw_state = torch.load(weight_path, map_location="cpu")
if isinstance(raw_state, dict) and "state_dict" in raw_state:
    raw_state = raw_state["state_dict"]
if not isinstance(raw_state, dict):
    raise TypeError("Весовой файл ДЗ 3 должен содержать state_dict.")
raw_state = {str(k).removeprefix("module."): v for k, v in raw_state.items()}
torch.save(
    {"state_dict": raw_state, "config": DZ3_MODEL_CONFIG},
    os.path.join(ART, "table_extractor.pt"),
)
with open(os.path.join(ART, "model_config.json"), "w", encoding="utf-8") as f:
    json.dump(DZ3_MODEL_CONFIG, f, ensure_ascii=False, indent=2)

print("в архив кладем модель:", best_model_name)
print("архитектура:", DZ3_MODEL_CONFIG["architecture"])

for src in [
    "/content/metrics_dz3.json",
    "/content/experiments.csv",
    "/content/loss_comparison.csv",
    "/content/sam_comparison.csv",
    "/content/crops.csv",
    "/content/transfer_check.csv",
    "/content/sam_vs_unet.png",
    "/content/crops_preview.png",
    "/content/transfer_check.png",
    "/content/encoder_comparison.csv",
    "/content/encoder_comparison.png",
]:
    if os.path.exists(src):
        shutil.copy(src, ART)

# Несколько вырезанных таблиц — образец результата.
os.makedirs(f"{ART}/crops_sample", exist_ok=True)
for _, row in crops_df.head(10).iterrows():
    shutil.copy(f"{CROPS_DIR}/{row['crop']}", f"{ART}/crops_sample/{row['crop']}")

# Артефакты ДЗ 2 сохраняем для прослеживаемости. Основной ДЗ 4 все равно
# принимает отдельный dz2_artifacts.zip как самостоятельный результат студента.
dz2_files = {
    "classifier.pt": "dz2_classifier.pt",
    "best_model.pt": "dz2_best_model.pt",
    "model_config.json": "dz2_model_config.json",
    "artifact_manifest.json": "dz2_artifact_manifest.json",
    "metrics.json": "dz2_metrics.json",
    "clean_labels.csv": "dz2_clean_labels.csv",
    "train.txt": "dz2_train.txt",
    "val.txt": "dz2_val.txt",
    "fit_train.txt": "dz2_fit_train.txt",
    "tuning.txt": "dz2_tuning.txt",
    "holdout.txt": "dz2_holdout.txt",
}
for source_name, target_name in dz2_files.items():
    src = DZ2 / source_name
    if src.exists():
        shutil.copy(src, os.path.join(ART, target_name))

with open(os.path.join(ART, "requirements.txt"), "w", encoding="utf-8") as f:
    f.write("torch==" + torch.__version__.split("+")[0] + "\n")
    f.write("albumentations==" + A.__version__ + "\n")
    f.write("opencv-python==" + cv2.__version__ + "\n")
    f.write("segmentation-models-pytorch==" + smp.__version__ + "\n")
    f.write("segment-anything\n")
    f.write("timm==" + timm.__version__ + "\n")

with open(os.path.join(ART, "artifact_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(
        {
            "schema_version": 2,
            "assignment": "DZ3",
            "provides": ["table_extractor", "table_crop_postprocessing"],
            "consumed_by": "DZ4",
            "primary_checkpoint": "table_extractor.pt",
            "legacy_state_dict": "best_unet.pt",
            "model_config": "model_config.json",
            "metrics": "metrics_dz3.json",
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

shutil.make_archive("/content/dz3_artifacts", "zip", ART)
print(f"готово: dz3_artifacts.zip ({os.path.getsize('/content/dz3_artifacts.zip') / 1e6:.1f} МБ)")
print("содержимое:", sorted(os.listdir(ART)))
print("Скачайте архив и СОХРАНИТЕ: ДЗ 4 загрузит модель без повторного обучения.")

from google.colab import files
files.download("/content/dz3_artifacts.zip")


## Шаг 9. Рефлексия и сдача

### ✍️ Рефлексия ПОСЛЕ выполнения

**1. Что было задано и что получилось** (2—3 предложения; сводка из `metrics_dz3.json` — ваш фактический материал):

✍️ Ваш ответ: ...

**2. Метрики.** Почему попиксельная точность оказалась бесполезной и какое число вы считаете честным ответом заказчику:

✍️ Ваш ответ: ...

**3. Функция потерь.** Что дала замена BCE на Dice и почему разница получилась именно такой:

✍️ Ваш ответ: ...

**4. SAM.** В каком сценарии вы возьмете фундаментальную модель, а в каком свою:

✍️ Ваш ответ: ...

**5. Перенос.** Что показала проверка на данных заказчика и что нужно, чтобы работало лучше:

✍️ Ваш ответ: ...

**6. Использование ИИ**: цель, промпты, где помощник ошибся:

✍️ Ваш ответ: ...

**7. Самооценка**: что бы вы сделали иначе, будь у вас еще неделя:

✍️ Ваш ответ: ...

### Перед сдачей
1. «Среда выполнения» → «Перезапустить и выполнить все»: ноутбук проходит без ошибок.
2. Все отчетные ячейки ✍️ и рефлексия заполнены вашими словами и числами.
3. Три текстовых поля в `RESULTS` заполнены.
4. Архив `dz3_artifacts.zip` содержит `table_extractor.pt`, `model_config.json`, финальный порог и скачан для ДЗ 4.


Спасибо за работу! В ДЗ 4 все собранное поедет на видео: документы будут проезжать перед камерой, и конвейер должен успевать за потоком.